# ETL: CSV `landing-zone` -> Parquet `raw` (via Trino)

In [1]:
import pandas as pd
import boto3
import io
import os
import s3fs
from sqlalchemy import create_engine

import warnings
warnings.filterwarnings('ignore')

# MinIO Configs
MINIO_ENDPOINT = os.getenv('AWS_S3_ENDPOINT', 'http://minio:9000')
MINIO_ACCESS_KEY = os.getenv('AWS_ACCESS_KEY_ID', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('AWS_SECRET_ACCESS_KEY', 'minioadmin')

print("Configurando cliente S3/MinIO...")
s3_client = boto3.client(
    's3', endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name='us-east-1'
)

Configurando cliente S3/MinIO...


## 1. Extract (Extração)

In [2]:
# 1. Read from landing-zone
landing_bucket = 'landing-zone'
file_name = 'penguins.csv'

print(f"Extraindo {file_name} da {landing_bucket}...")
response = s3_client.get_object(Bucket=landing_bucket, Key=file_name)
df = pd.read_csv(io.BytesIO(response['Body'].read()))

print(f"Dados extraídos com sucesso! Linhas: {df.shape[0]}")
df.head(3)

Extraindo penguins.csv da landing-zone...
Dados extraídos com sucesso! Linhas: 344


,especies,ilha,bico_comp_mm,bico_largura_mm,nadadeira_comp_mm,masso_corporal_g,massa_corporal_kg,sexo,ano
0,adelie,torgersen,39.1,18.7,181.0,3750.0,3.75,macho,2007
1,adelie,torgersen,39.5,17.4,186.0,3800.0,3.80,femea,2007
2,adelie,torgersen,40.3,18.0,195.0,3250.0,3.25,femea,2007


## 2. Salvar como Parquet Bruto (Raw)

In [3]:
# 2. Transform to Parquet and Load to 'raw' bucket
# We use s3fs to write Pandas DataFrame directly as Parquet into MinIO
raw_bucket = 'raw'
s3_path = f"s3://{raw_bucket}/penguins/penguins_data.parquet"

print(f"Convertendo para Parquet e salvando no bucket '{raw_bucket}'...")
s3_fs = s3fs.S3FileSystem(
    key=MINIO_ACCESS_KEY,
    secret=MINIO_SECRET_KEY,
    client_kwargs={'endpoint_url': MINIO_ENDPOINT}
)

df.to_parquet(s3_path, filesystem=s3_fs, index=False)
print("Arquivo Parquet salvo com sucesso!")

Convertendo para Parquet e salvando no bucket 'raw'...


FileNotFoundError: The specified bucket does not exist

## 3. Load / Catálogo no Trino (Redshift Simulator)

In [ ]:
# 3. Represent in Trino (Database Raw, Table Penguins)
from sqlalchemy import text

# Conectando ao Trino via JDBC/SQLAlchemy
trino_engine = create_engine('trino://jovyan@trino:8080/minio')

with trino_engine.connect() as con:
    # Cria o Schema (Database) apontando a location pro S3
    print("Criando Schema 'raw'...")
    con.execute(text("CREATE SCHEMA IF NOT EXISTS minio.raw WITH (location = 's3://raw/')"))
    
    # Para registrar nossa tabela, usamos o pandas to_sql que se integra
    # perfeitamente aos catalogos do Iceberg/Hive para criar e popular!
    print("Registrando e inserindo dados na Tabela 'penguins' dentro do database 'raw'...")
    
    # O comando to_sql sobrescreve na engine Trino e salva em formato columnar Parquet nativamente!
    # Caso já exista, ele deleta e recria.
    df.to_sql('penguins', con=trino_engine, schema='raw', if_exists='replace', index=False, method='multi', chunksize=1000)
    
    print("Tabela minio.raw.penguins registrada e pronta para consultas Analytics!")

In [ ]:
# 4. Homologação / Teste de Query Analytics
with trino_engine.connect() as con:
    result = con.execute(text("SELECT especies, count(*) as count FROM minio.raw.penguins GROUP BY especies"))
    print("\nResultado Consulta SQL Direto do Trino:")
    for row in result:
        print(row)